In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql.window import Window
from pyspark.sql import functions as F

In [0]:
df = spark.read.table('fintech360.silver.loans')

In [0]:
%sql
select * from fintech360.silver.loans limit 5;

In [0]:

from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType

# --------------------------------------------------
# STEP 1: CONFIGURATION
# --------------------------------------------------
silver_table = "silver.loans_validated"
quarantine_table = "bronze.loans_quarantine"

valid_statuses = [
    "APPLIED", "APPROVED", "REJECTED",
    "DISBURSED", "CLOSED", "CANCELLED",
    "DEFAULTED", "WRITTEN_OFF"
]

# --------------------------------------------------
# STEP 2: DETECT DUPLICATE LOAN IDS
# --------------------------------------------------
duplicate_ids = (
    df.filter(F.col("loan_id").isNotNull())
      .groupBy("loan_id")
      .count()
      .filter(F.col("count") > 1)
      .select("loan_id")
      .withColumn("_duplicate_loan_id", F.lit(True))
)

df_check = df.join(duplicate_ids, "loan_id", "left")

# --------------------------------------------------
# STEP 3: VALIDATE JSON
# --------------------------------------------------
json_schema = StructType([
    StructField("application_channel", StringType(), True)
])

df_check = df_check.withColumn(
    "_parsed_json",
    F.from_json(F.col("loan_details_json"), json_schema)
)

# --------------------------------------------------
# STEP 4: DEFINE QUALITY RULES
# Each failed rule adds an issue to the array
# --------------------------------------------------
issues = [
    F.when(F.col("loan_id").isNull(), "MISSING_LOAN_ID"),

    F.when(F.col("customer_id").isNull(), "MISSING_CUSTOMER_ID"),

    F.when(
        F.col("application_id").isNull() |
        (F.trim(F.col("application_id")) == ""),
        "MISSING_APPLICATION_ID"
    ),

    F.when(
        F.col("application_ts").isNull(),
        "MISSING_OR_INVALID_APPLICATION_TIMESTAMP"
    ),

    F.when(
        F.col("principal_amount").isNull() |
        (F.col("principal_amount") <= 0),
        "INVALID_PRINCIPAL_AMOUNT"
    ),

    F.when(
        F.col("annual_interest_rate_pct").isNull() |
        (F.col("annual_interest_rate_pct") < 0) |
        (F.col("annual_interest_rate_pct") > 100),
        "INVALID_INTEREST_RATE"
    ),

    F.when(
        F.col("tenure_months").isNull() |
        (F.col("tenure_months") <= 0),
        "INVALID_TENURE"
    ),

    F.when(
        F.col("loan_status").isNull() |
        (~F.col("loan_status").isin(valid_statuses)),
        "INVALID_LOAN_STATUS"
    ),

    F.when(
        F.col("loan_source").isNull() |
        (F.trim(F.col("loan_source")) == ""),
        "MISSING_LOAN_SOURCE"
    ),

    F.when(
        F.col("currency").isNull() |
        (F.col("currency") != "INR"),
        "INVALID_CURRENCY"
    ),

    F.when(
        (F.col("loan_status") == "DISBURSED") &
        F.col("disbursement_ts").isNull(),
        "DISBURSED_WITHOUT_TIMESTAMP"
    ),

    F.when(
        F.col("disbursement_ts").isNotNull() &
        F.col("application_ts").isNotNull() &
        (F.col("disbursement_ts") < F.col("application_ts")),
        "DISBURSEMENT_BEFORE_APPLICATION"
    ),

    F.when(
        F.col("maturity_date").isNotNull() &
        F.col("disbursement_ts").isNotNull() &
        (F.col("maturity_date") <= F.col("disbursement_ts")),
        "INVALID_MATURITY_DATE"
    ),

    F.when(
        F.col("created_at").isNotNull() &
        F.col("application_ts").isNotNull() &
        (F.col("created_at") < 1000000000),
        "CREATED_AT_SUSPICIOUS"
    ),

    F.when(
        F.col("updated_at").isNotNull() &
        F.col("created_at").isNotNull() &
        (F.col("updated_at") < F.col("created_at")),
        "UPDATED_BEFORE_CREATED"
    ),

    F.when(
        F.col("_duplicate_loan_id").isNotNull(),
        "DUPLICATE_LOAN_ID"
    ),

    F.when(
        F.col("loan_details_json").isNotNull() &
        F.col("_parsed_json").isNull(),
        "MALFORMED_LOAN_JSON"
    ),

    F.when(
        F.col("_rescued_data").isNotNull() &
        (F.trim(F.col("_rescued_data")) != ""),
        "SCHEMA_RESCUED_DATA_PRESENT"
    )
]

# --------------------------------------------------
# STEP 5: ASSIGN PASS / FAIL
# --------------------------------------------------
df_quality = (
    df_check
    .withColumn(
        "quality_issues",
        F.filter(F.array(*issues), lambda x: x.isNotNull())
    )
    .withColumn(
        "quality_status",
        F.when(F.size("quality_issues") == 0, "PASS")
         .otherwise("FAIL")
    )
    .withColumn("quality_checked_at", F.current_timestamp())
)

# --------------------------------------------------
# STEP 6: SPLIT PASS AND FAIL RECORDS
# --------------------------------------------------
helper_cols = ["_duplicate_loan_id", "_parsed_json"]

silver_ready = (
    df_quality
    .filter(F.col("quality_status") == "PASS")
    .drop(
        *helper_cols,
        "quality_issues",
        "quality_status",
        "quality_checked_at"
    )
)

quarantine = (
    df_quality
    .filter(F.col("quality_status") == "FAIL")
    .drop(*helper_cols)
)

# --------------------------------------------------
# STEP 7: WRITE BOTH OUTPUTS
# Snapshot/batch example: replace previous table contents
# --------------------------------------------------
silver_ready.write.format("delta").mode("overwrite").saveAsTable(
    silver_table
)

quarantine.write.format("delta").mode("overwrite").saveAsTable(
    quarantine_table
)

# --------------------------------------------------
# STEP 8: VERIFY OUTPUT
# --------------------------------------------------
print("Silver-ready records:", silver_ready.count())
print("Quarantined records:", quarantine.count())

display(
    df_quality.groupBy("quality_status").count()
)

display(
    quarantine.select(
        "loan_id",
        "customer_id",
        "loan_status",
        "quality_issues"
    ).limit(50)
)


In [0]:
silver_ready.write.format("delta").mode("overwrite").saveAsTable(
    'fintech360.data_quality_check.loans')